# MRAG Ablation Study

Leave-one-out ablations on **Fable 5 fewshot / MUTCD-150 frozen**. Six components disabled one at a time; each Δ vs baseline (88.08) reported with paired bootstrap 95% CI.

**Runs entirely in this notebook.** Does not touch the main `mrag/` package — component toggles happen at runtime via monkey-patches in `ablations/ablation_shims.py`.

**Order of operations:**
1. Env setup (mount + clone + install)
2. Configure — pick ablations to run and set baseline path
3. (Optional) Cheap retrieval-only pass first — minutes per ablation, use to deprioritize dead components
4. Full sweep — ~70 min per ablation × 6 = ~7h wallclock
5. Score via GPT-5.6 rubric — ~30 min per ablation × 6 = ~3h
6. Analyze — paired bootstrap CI, summary CSV, forest plot

Resume-safe throughout — kill and restart at any point.


## 1. Environment setup

In [ ]:
# Colab setup: mount Drive, clone repo, install deps.
# Skip cells you've already run in a live kernel.

import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
print('Colab detected:', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    else:
        print('Drive already mounted')


In [ ]:
# Clone / update MRAG_stp2 repo.
REPO_URL = 'https://github.com/hannanazad/MRAG_stp2.git'
REPO_DIR = '/content/MRAG_stp2'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Repo ready at', REPO_DIR)


In [ ]:
# Install extra deps for the ablation harness.
# The main mrag/ requirements are handled by the existing pipeline notebook.
!pip install -q pyyaml openai matplotlib numpy


In [ ]:
# API keys — read from Colab Secrets when available.
try:
    from google.colab import userdata
    for k in ('OPENAI_API_KEY', 'DASHSCOPE_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY'):
        try:
            v = userdata.get(k)
            if v:
                os.environ[k] = v
                print(f'  {k}: loaded from Colab Secrets')
        except Exception:
            pass
except ImportError:
    print('Not in Colab; expecting env vars already set.')

assert os.environ.get('OPENAI_API_KEY'), 'OPENAI_API_KEY required for scoring.'
print('OPENAI_API_KEY is set. Length:', len(os.environ['OPENAI_API_KEY']))


## 2. Configure the run

In [ ]:
# --- CHANGE THESE ---
GOLD_QA_PATH        = '/content/drive/MyDrive/MRAG/eval/gold_qa.jsonl'
BASELINE_SCORED_PATH = '/content/drive/MyDrive/MRAG/eval/scored.jsonl'  # your MUTCD-150 Fable-5 fewshot baseline
RESULTS_DIR         = '/content/drive/MyDrive/MRAG/eval/ablations'      # where per-ablation results are saved

# Which ablations to run this pass. Comment out to skip.
ABLATIONS_TO_RUN = [
    'A1_no_router',
    'A2_no_vlm_filter',
    'A3_no_graph',
    'A4_no_rule_type',
    'A5_no_hierarchy',
    'A6_no_reranker',
]

# CHEAP FIRST PASS: retrieval-only (no generation, no scoring). Set True to
# get retrieval metrics per ablation in minutes each. Use to identify
# dead components before committing to the full generation loop.
RETRIEVAL_ONLY = False

# Model / prompt style — frozen for the study.
VLM_ALIAS    = 'fable'
PROMPT_STYLE = 'fewshot'

# For quick smoke tests, set to e.g. 5. None = full 150.
LIMIT_N_QUESTIONS = None

# --- prepare paths ---
from pathlib import Path
RESULTS_DIR = Path(RESULTS_DIR)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Results dir:', RESULTS_DIR)
print('Ablations:', ABLATIONS_TO_RUN)


In [ ]:
# Sanity-check the gold file and baseline before spending any tokens.
import json
def peek_jsonl(path, n=1):
    rows = []
    with open(path) as f:
        for i, line in enumerate(f):
            if i >= n: break
            rows.append(json.loads(line))
    return rows

def count_jsonl(path):
    return sum(1 for _ in open(path))

print('gold_qa.jsonl rows:', count_jsonl(GOLD_QA_PATH))
print('first gold record keys:', list(peek_jsonl(GOLD_QA_PATH)[0].keys()))

if Path(BASELINE_SCORED_PATH).exists():
    print('baseline rows:', count_jsonl(BASELINE_SCORED_PATH))
    print('first baseline record keys:', list(peek_jsonl(BASELINE_SCORED_PATH)[0].keys()))
else:
    print('WARNING: baseline not found at', BASELINE_SCORED_PATH)
    print('You can still run + score ablations, but analyze_ablations.py needs this file.')


## 3. Initialize the pipeline (once)

In [ ]:
# Freeze VLM + answer style, init pipeline.
from mrag.config import CFG
CFG.set_vlm_model(VLM_ALIAS)
CFG.set_answer_style(PROMPT_STYLE)
print('VLM:', getattr(CFG, 'vlm_model_api', '?'))
print('Answer style:', getattr(CFG, 'prompt_style_answer', '?'))

# Adjust the import if init_pipeline lives elsewhere in your tree.
try:
    from mrag.pipeline import init_pipeline
except ImportError:
    # fallback locations
    try:
        from mrag.ask import init_pipeline  # sometimes lives here
    except ImportError as e:
        raise ImportError('init_pipeline() not found in mrag.pipeline or mrag.ask — patch this cell') from e

pipeline = init_pipeline()
print('Pipeline ready:', type(pipeline).__name__)
print('Reranker type:', type(getattr(pipeline, 'reranker', None)).__name__)


In [ ]:
# Verify ablation shims are importable and show what each does.
sys.path.insert(0, REPO_DIR)  # ensures `ablations` package is on path
from ablations.ablation_shims import ABLATIONS, list_ablations
for a in list_ablations():
    print(f'  {a["id"]:20s} — {a["description"]}')


## 4. Smoke test (recommended: 5 questions, ~2 min)

In [ ]:
# Run A1 on the first 5 questions to confirm end-to-end wiring before
# committing to the full 6×150 sweep.
from ablations.run_ablation import run_ablation_sweep
smoke_id = 'A1_no_router'
smoke_output = RESULTS_DIR / f'SMOKE_runs_{smoke_id}.jsonl'
if smoke_output.exists():
    smoke_output.unlink()  # fresh run

run_ablation_sweep(
    ablation_id=smoke_id,
    gold_path=Path(GOLD_QA_PATH),
    output_path=smoke_output,
    vlm_alias=VLM_ALIAS,
    prompt_style=PROMPT_STYLE,
    limit=5,
    retrieval_only=RETRIEVAL_ONLY,
)

# Inspect one record.
with open(smoke_output) as f:
    for line in f:
        rec = json.loads(line)
        print(rec['qa_id'], '  error=', rec.get('error'), '  latency=', round(rec.get('latency_s', 0), 1), 's')
        break


## 5. Full sweep (all selected ablations × 150 questions)

Resume-safe. Kill and restart at any point; only unfinished / errored qa_ids will be re-run.

Expected wallclock: ~70 min per ablation with generation; minutes per ablation if `RETRIEVAL_ONLY=True`.

In [ ]:
# Run each selected ablation to completion, sequentially.
for ablation_id in ABLATIONS_TO_RUN:
    output_path = RESULTS_DIR / f'runs_{ablation_id}.jsonl'
    print(f'\n{"="*60}\n{ablation_id}  →  {output_path}\n{"="*60}')
    run_ablation_sweep(
        ablation_id=ablation_id,
        gold_path=Path(GOLD_QA_PATH),
        output_path=output_path,
        vlm_alias=VLM_ALIAS,
        prompt_style=PROMPT_STYLE,
        limit=LIMIT_N_QUESTIONS,
        retrieval_only=RETRIEVAL_ONLY,
    )

print('\nAll requested sweeps complete.')


## 6. Score every ablation via GPT-5.6 rubric

Skip this step if `RETRIEVAL_ONLY=True` (retrieval metrics are computed inside the sweep + analyze step).

Expected wallclock: ~30 min per ablation.

In [ ]:
if RETRIEVAL_ONLY:
    print('RETRIEVAL_ONLY=True — skipping generation scoring.')
else:
    from ablations.score_ablation import score_runs
    for ablation_id in ABLATIONS_TO_RUN:
        runs_path = RESULTS_DIR / f'runs_{ablation_id}.jsonl'
        out_path  = RESULTS_DIR / f'scored_{ablation_id}.jsonl'
        if not runs_path.exists():
            print(f'skip {ablation_id}: {runs_path} not found')
            continue
        print(f'\n{"="*60}\n{ablation_id}: scoring {runs_path}\n{"="*60}')
        score_runs(
            runs_path=runs_path,
            gold_path=Path(GOLD_QA_PATH),
            output_path=out_path,
        )
    print('\nAll scoring complete.')


## 7. Analyze: paired bootstrap CI vs baseline

In [ ]:
from ablations.analyze_ablations import analyze
import glob

summary_csv = RESULTS_DIR / 'summary.csv'
deltas_out  = RESULTS_DIR / 'per_item_deltas.jsonl'
plot_out    = RESULTS_DIR / 'delta_plot.png'

scored_paths = [Path(p) for p in sorted(glob.glob(str(RESULTS_DIR / 'scored_*.jsonl')))]
print('Scored files:', [p.name for p in scored_paths])

if not scored_paths:
    print('No scored files. Run cell 6 first (or set RETRIEVAL_ONLY=False).')
elif not Path(BASELINE_SCORED_PATH).exists():
    print(f'Baseline not found at {BASELINE_SCORED_PATH}. Set BASELINE_SCORED_PATH in cell 2.')
else:
    analyze(
        baseline_path=Path(BASELINE_SCORED_PATH),
        ablation_paths=scored_paths,
        summary_path=summary_csv,
        deltas_path=deltas_out,
        plot_path=plot_out,
        B=10000,
    )


In [ ]:
# Display the summary table.
import csv
if summary_csv.exists():
    with open(summary_csv) as f:
        for row in csv.DictReader(f):
            print(f"{row['ablation']:20s}  "
                  f"Δ={float(row['mean_delta']):+7.3f}  "
                  f"CI=[{float(row['ci_low_95']):+7.3f}, {float(row['ci_high_95']):+7.3f}]  "
                  f"p={float(row['p_two_sided']):.3f}  "
                  f"sig={row['sig_at_05']}")
else:
    print('summary.csv not found yet.')


In [ ]:
# Show the forest plot inline.
from IPython.display import Image, display
if plot_out.exists():
    display(Image(filename=str(plot_out)))
else:
    print('delta_plot.png not produced.')


## 8. Interpret

**Positive Δ** means the ablation *helped* — that component was hurting on this benchmark. Sign to investigate.

**Negative Δ with CI excluding 0** means the component contributes meaningfully — remove it and you lose score.

**CI straddles 0** means the component's effect is not distinguishable from noise on 150 questions. Not evidence of no effect, but not evidence of one either — worth a larger benchmark before removing.

**Next steps (per your MRAG_20 KG):**
- If A6 (reranker) Δ is small, consider dropping it for the ~30% latency saving
- If A2 (VLM filter) Δ is small, `vlm_model_filter_idea` becomes less urgent
- If A1 (router) Δ is large and negative, that's your headline ablation result
